# 1627. Graph Connectivity With Threshold

## Topic Alignment
- **Role Relevance**: Model mathematical relationships in network connectivity where nodes are connected based on divisibility constraints.
- **Scenario**: Analyze data relationships with threshold-based filtering, version compatibility systems, or hierarchical dependencies where parent-child relationships follow divisibility rules.

## Metadata Summary
- Source: [Graph Connectivity With Threshold](https://leetcode.com/problems/graph-connectivity-with-threshold/)
- Tags: `Union-Find`, `Math`, `Array`
- Difficulty: Hard
- Recommended Priority: High

## Problem Statement
We have `n` cities labeled from `1` to `n`. Two different cities with labels `x` and `y` are directly connected by a bidirectional road if and only if `x` and `y` share a common divisor **strictly greater** than some `threshold`. More formally, cities with labels `x` and `y` have a road between them if there exists an integer `z` such that all of the following are true:
- `x % z == 0`
- `y % z == 0`
- `z > threshold`

Given the integers `n` and `threshold`, and an array of `queries`, return an array of booleans `answer`, where `answer[i]` is `true` if cities `queries[i][0]` and `queries[i][1]` are connected directly or indirectly (i.e., there is some path between them), otherwise `answer[i]` is `false`.

**Example 1**:
```
Input: n = 6, threshold = 2, queries = [[1,4],[2,5],[3,6]]
Output: [false,false,true]
Explanation:
- Cities 3 and 6 share divisor 3 > 2, so connected.
- 1 and 4 share only 1 ≤ 2.
- 2 and 5 share only 1 ≤ 2.
```

**Example 2**:
```
Input: n = 6, threshold = 0, queries = [[4,5],[3,4],[3,2],[2,6],[1,3]]
Output: [true,true,true,true,true]
```

## Progressive Hints
- Hint 1: Instead of checking all pairs, iterate through each possible divisor z > threshold.
- Hint 2: For each divisor z, connect all multiples of z (z, 2z, 3z, ...) that are ≤ n.
- Hint 3: Use Union-Find to efficiently group cities sharing common divisors.
- Hint 4: Process divisors from threshold+1 to n/2, connecting each divisor with its multiples.

## Solution Overview
Build a Union-Find structure by iterating through all possible divisors > threshold and connecting all their multiples. Then answer queries by checking if two cities are in the same component.

## Detailed Explanation
1. **Key Insight**: Two cities share a common divisor z > threshold if both are multiples of z.
   - Instead of checking all city pairs (O(n²)), iterate through divisors (O(n log n))
2. **Building Connections**: For each divisor z from (threshold + 1) to n:
   - Find all multiples: z, 2z, 3z, ... up to n
   - Connect consecutive multiples: union(z, 2z), union(2z, 3z), etc.
   - This creates a component of all cities divisible by z
3. **Optimization**: We can connect z with all its multiples directly:
   - union(z, 2z), union(z, 3z), union(z, 4z), ...
   - Achieves same result with cleaner code
4. **Query Processing**: For each query [x, y]:
   - Simply check: find(x) == find(y)
   - O(α(n)) per query with path compression
5. **Mathematical Property**: If cities share any divisor > threshold, they'll be connected through that divisor's component.
6. **Edge Cases**: threshold = 0 connects all cities (divisor 1 doesn't count, but 2, 3, ... connect many), threshold ≥ n/2 means few/no connections.

## Complexity Trade-off Table
| Approach | Time Complexity | Space Complexity | Notes |
| --- | --- | --- | --- |
| Union-Find with divisors | O(n log n + q α(n)) | O(n) | Optimal for multiple queries |
| Precompute all pairs | O(n² log n) | O(n²) | Too slow and memory-intensive |
| BFS per query | O(q * n²) | O(n²) | Extremely slow for many queries |
| GCD per query pair | O(q * n * log n) | O(n) | Misses indirect connections |

## Reference Implementation

In [ ]:
from typing import List


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n + 1))  # Cities labeled 1 to n
        self.rank = [0] * (n + 1)
    
    def find(self, x: int) -> int:
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]
    
    def union(self, x: int, y: int):
        root_x, root_y = self.find(x), self.find(y)
        if root_x != root_y:
            # Union by rank
            if self.rank[root_x] > self.rank[root_y]:
                self.parent[root_y] = root_x
            elif self.rank[root_x] < self.rank[root_y]:
                self.parent[root_x] = root_y
            else:
                self.parent[root_y] = root_x
                self.rank[root_x] += 1
    
    def connected(self, x: int, y: int) -> bool:
        return self.find(x) == self.find(y)


def areConnected(n: int, threshold: int, queries: List[List[int]]) -> List[bool]:
    uf = UnionFind(n)
    
    # For each possible divisor greater than threshold
    for z in range(threshold + 1, n + 1):
        # Connect all multiples of z
        # Start from z, then 2z, 3z, ...
        first_multiple = z
        multiple = 2 * z
        
        while multiple <= n:
            uf.union(first_multiple, multiple)
            multiple += z
    
    # Answer queries
    result = []
    for x, y in queries:
        result.append(uf.connected(x, y))
    
    return result

## Validation

In [ ]:
assert areConnected(6, 2, [[1,4],[2,5],[3,6]]) == [False, False, True]
assert areConnected(6, 0, [[4,5],[3,4],[3,2],[2,6],[1,3]]) == [True, True, True, True, True]
assert areConnected(5, 1, [[4,5],[3,4],[3,2],[2,4],[1,4]]) == [True, True, True, True, False]
assert areConnected(10, 0, [[1,2],[3,4]]) == [True, True]
assert areConnected(10, 5, [[1,10],[5,10],[6,9]]) == [False, False, True]
print('All tests passed for LC 1627.')

## Complexity Analysis
- Time Complexity: O(n log n + q α(n)), where n is number of cities, q is number of queries, and α is inverse Ackermann function.
  - Building Union-Find: O(n log n) - for each divisor z, we process n/z multiples, sum is harmonic series
  - Processing queries: O(q α(n)) - nearly constant time per query
- Space Complexity: O(n) for Union-Find structure.
- Bottleneck: Building connections through divisors (one-time preprocessing).

## Edge Cases & Pitfalls
- threshold = 0: Divisor 2 connects all even numbers, divisor 3 connects all multiples of 3, etc.
- threshold ≥ n: No connections possible (no divisor > threshold exists in range).
- Query with same city: Should return true (city connected to itself).
- Large threshold: Reduces number of connections significantly.
- Prime numbers > threshold: Only connect with their multiples.

## Follow-up Variants
- Dynamic threshold: Handle queries with different threshold values.
- Add/remove cities dynamically and maintain connectivity.
- Find size of connected component for a given city.
- Count total number of connected components.
- Find minimum threshold to connect two specific cities.

## Takeaways
- Iterate through property values (divisors) rather than all element pairs for efficiency.
- Divisibility relationships create natural groupings suitable for Union-Find.
- Preprocessing with Union-Find enables O(1) query responses for connectivity.
- Harmonic series sum (1 + 1/2 + 1/3 + ... + 1/n) ≈ O(log n) appears in divisor-based problems.
- Mathematical insights about divisibility reduce complexity from O(n²) to O(n log n).

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| 952 | Largest Component Size by Common Factor | Union-Find with prime factors |
| 547 | Number of Provinces | Basic Union-Find connectivity |
| 721 | Accounts Merge | Union-Find grouping |
| 1319 | Number of Operations to Make Network Connected | Union-Find with counting |